In [1]:
# ============================================================
# MIMICEL -> ProcessGAN Input Converter
# Input : ./data/mimicel_train.csv / val / test
# Output: ./results/encoded
# ============================================================

from pathlib import Path
import json
import pandas as pd


# ------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------

DATA_DIR = Path("./data")
OUT_DIR = Path("./results/encoded")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CASE_COL = "stay_id"
TIME_COL = "timestamps"
ACT_COL = "activity"

SPLITS = {
    "train": DATA_DIR / "mimicel_train.csv",
    "val": DATA_DIR / "mimicel_val.csv",
    "test": DATA_DIR / "mimicel_test.csv",
}


# ------------------------------------------------------------
# 1. Load all splits
# ------------------------------------------------------------

dfs = {}

for split, path in SPLITS.items():
    if not path.exists():
        print(f"[SKIP] {split}: file not found -> {path}")
        continue

    print(f"[LOAD] {split}: {path}")

    df = pd.read_csv(
        path,
        usecols=[CASE_COL, TIME_COL, ACT_COL],
        low_memory=False
    )

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[CASE_COL, TIME_COL, ACT_COL])
    df = df.sort_values([CASE_COL, TIME_COL]).reset_index(drop=True)

    dfs[split] = df

if "train" not in dfs:
    raise FileNotFoundError(
        "ProcessGAN 학습에는 ./data/mimicel_train.csv가 필요합니다."
    )


# ------------------------------------------------------------
# 2. Build activity vocabulary from TRAIN only
# ------------------------------------------------------------

print("[VOCAB] Building activity vocabulary from train split...")

train_activities = sorted(dfs["train"][ACT_COL].unique())

activity2id = {
    act: idx + 1
    for idx, act in enumerate(train_activities)
}

id2activity = {
    idx: act
    for act, idx in activity2id.items()
}

with open(OUT_DIR / "activity2id.json", "w", encoding="utf-8") as f:
    json.dump(activity2id, f, ensure_ascii=False, indent=2)

with open(OUT_DIR / "id2activity.json", "w", encoding="utf-8") as f:
    json.dump(id2activity, f, ensure_ascii=False, indent=2)

print(f"Number of activities: {len(activity2id)}")


# ------------------------------------------------------------
# 3. Time normalization reference from TRAIN only
# ------------------------------------------------------------

print("[TIME] Calculating train duration min/max...")

train_duration = (
    dfs["train"]
    .groupby(CASE_COL)[TIME_COL]
    .agg(["min", "max"])
)

train_duration["duration_seconds"] = (
    train_duration["max"] - train_duration["min"]
).dt.total_seconds()

duration_min = train_duration["duration_seconds"].min()
duration_max = train_duration["duration_seconds"].max()

if duration_max == duration_min:
    duration_max = duration_min + 1.0

time_meta = {
    "duration_min_seconds": float(duration_min),
    "duration_max_seconds": float(duration_max)
}

with open(OUT_DIR / "time_normalization_meta.json", "w", encoding="utf-8") as f:
    json.dump(time_meta, f, ensure_ascii=False, indent=2)

print(time_meta)


# ------------------------------------------------------------
# 4. Converter function
# ------------------------------------------------------------

def convert_split_to_processgan(df, split_name):
    """
    Output files:
    - {split}_MIMICEL.txt
    - {split}_MIMICEL_time_dif_norm.txt
    - {split}_MIMICEL_time_duration_norm.txt
    """

    seq_lines = []
    time_dif_lines = []
    duration_lines = []

    unknown_activity_count = 0
    skipped_cases = 0

    for case_id, g in df.groupby(CASE_COL, sort=False):
        g = g.sort_values(TIME_COL)

        activities = g[ACT_COL].tolist()
        times = g[TIME_COL].tolist()

        # train vocab에 없는 activity는 skip
        if any(act not in activity2id for act in activities):
            unknown_activity_count += 1
            continue

        if len(activities) < 2:
            skipped_cases += 1
            continue

        token_seq = [activity2id[act] for act in activities]

        start_time = times[0]
        end_time = times[-1]

        duration_sec = (end_time - start_time).total_seconds()

        if duration_sec < 0:
            skipped_cases += 1
            continue

        # duration이 0이면 inter-event 계산이 불안정하므로 1초로 보정
        duration_for_ratio = max(duration_sec, 1.0)

        # inter-event time difference
        diffs = [0.0]
        for i in range(1, len(times)):
            delta_sec = (times[i] - times[i - 1]).total_seconds()
            delta_sec = max(delta_sec, 0.0)
            diffs.append(delta_sec / duration_for_ratio)

        # duration normalization based on train duration min/max
        duration_norm = (
            (duration_sec - duration_min)
            / (duration_max - duration_min)
        )

        duration_norm = min(max(duration_norm, 0.0), 1.0)

        seq_lines.append(" ".join(map(str, token_seq)))
        time_dif_lines.append(" ".join(f"{x:.8f}" for x in diffs))
        duration_lines.append(f"{duration_norm:.8f}")

    # Save
    seq_path = OUT_DIR / f"{split_name}_MIMICEL.txt"
    dif_path = OUT_DIR / f"{split_name}_MIMICEL_time_dif_norm.txt"
    dur_path = OUT_DIR / f"{split_name}_MIMICEL_time_duration_norm.txt"

    seq_path.write_text("\n".join(seq_lines), encoding="utf-8")
    dif_path.write_text("\n".join(time_dif_lines), encoding="utf-8")
    dur_path.write_text("\n".join(duration_lines), encoding="utf-8")

    summary = {
        "split": split_name,
        "num_cases_written": len(seq_lines),
        "unknown_activity_cases_skipped": unknown_activity_count,
        "other_cases_skipped": skipped_cases,
        "seq_file": str(seq_path),
        "time_dif_file": str(dif_path),
        "duration_file": str(dur_path),
    }

    return summary


# ------------------------------------------------------------
# 5. Convert train / val / test
# ------------------------------------------------------------

summaries = []

for split_name, df in dfs.items():
    print(f"[CONVERT] {split_name}")
    summary = convert_split_to_processgan(df, split_name)
    summaries.append(summary)
    print(summary)


# ------------------------------------------------------------
# 6. Save summary
# ------------------------------------------------------------

summary_df = pd.DataFrame(summaries)
summary_df.to_csv(
    OUT_DIR / "processgan_encoding_summary.csv",
    index=False
)

print("\nDone.")
print(f"Outputs saved to: {OUT_DIR.resolve()}")

[LOAD] train: data\mimicel_train.csv
[LOAD] val: data\mimicel_val.csv
[LOAD] test: data\mimicel_test.csv
[VOCAB] Building activity vocabulary from train split...
Number of activities: 6
[TIME] Calculating train duration min/max...
{'duration_min_seconds': 60.0, 'duration_max_seconds': 7089240.0}
[CONVERT] train
{'split': 'train', 'num_cases_written': 297519, 'unknown_activity_cases_skipped': 0, 'other_cases_skipped': 0, 'seq_file': 'results\\encoded\\train_MIMICEL.txt', 'time_dif_file': 'results\\encoded\\train_MIMICEL_time_dif_norm.txt', 'duration_file': 'results\\encoded\\train_MIMICEL_time_duration_norm.txt'}
[CONVERT] val
{'split': 'val', 'num_cases_written': 42502, 'unknown_activity_cases_skipped': 0, 'other_cases_skipped': 0, 'seq_file': 'results\\encoded\\val_MIMICEL.txt', 'time_dif_file': 'results\\encoded\\val_MIMICEL_time_dif_norm.txt', 'duration_file': 'results\\encoded\\val_MIMICEL_time_duration_norm.txt'}
[CONVERT] test
{'split': 'test', 'num_cases_written': 85007, 'unknow